In [ ]:
#---------------------
# 0. 필요한 패키지 로드
#---------------------

import os, random
from glob import glob
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from typing import List, Tuple, Dict
from PIL import Image
from tqdm import trange   # 진행률 표시 (evaluate에서 사용)


In [ ]:
#---------------------
# 1. 크롭된 데이터 로드
#---------------------

# 데이터 경로
BASE_DIR = "./all_crops"

# 모든 ID 폴더 불러오기
all_ids = [d for d in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, d))]
print("전체 ID 개수:", len(all_ids))

# 랜덤하게 3000명 선택
selected_ids = random.sample(all_ids, 3000)

# Train/Test ID 분리 (ID 기준)
train_ids, test_ids = train_test_split(selected_ids, test_size=0.2, random_state=42)

print("Train ID 개수:", len(train_ids))
print("Test ID 개수:", len(test_ids))

# 함수: ID별로 이미지 샘플링 (최대 5장만)
def sample_images_for_id(id_path, n_samples=5):
    images = glob(os.path.join(id_path, "*.jpg")) + glob(os.path.join(id_path, "*.png"))
    if len(images) < n_samples:
        return images  # 부족하면 전부 사용
    return random.sample(images, n_samples)

# 예시: 트레인셋 구성
train_data = {}
for id_ in train_ids:
    id_path = os.path.join(BASE_DIR, id_)
    train_data[id_] = sample_images_for_id(id_path, n_samples=5)

# 예시: 테스트셋 구성
test_data = {}
for id_ in test_ids:
    id_path = os.path.join(BASE_DIR, id_)
    test_data[id_] = sample_images_for_id(id_path, n_samples=5)

print("샘플 Train ID 1개:", list(train_data.keys())[0])
print("해당 ID의 이미지 개수:", len(train_data[list(train_data.keys())[0]]))

In [ ]:
#---------------------
# 2. 데이터 전처리
#---------------------

IMG_SIZE = 224
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomApply([transforms.ColorJitter(0.2,0.2,0.2,0.1)], p=0.3),
    transforms.RandomRotation(3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

def load_img(path: str, tfm) -> torch.Tensor:
    img = Image.open(path).convert("RGB")
    return tfm(img)

# ID별에서 k장 뽑기 (부족하면 중복 허용) 
def sample_k(paths: List[str], k: int) -> List[str]:
    if len(paths) >= k:
        return random.sample(paths, k)
    # 부족하면 중복 허용 샘플
    return random.choices(paths, k=k)

In [ ]:
#---------------------
# 2. 데이터 전처리
#---------------------

def make_batch(id2paths: Dict[str, List[str]],
               batch_size: int = 16,
               k_refs: int = 5,
               pos_ratio: float = 0.5,
               train_mode: bool = True) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    returns:
      probe: (B, 3, H, W)
      refs : (B, K, 3, H, W)
      labels: (B,)  1=같은 ID, 0=다른 ID
    """
    ids = list(id2paths.keys())
    B_pos = int(batch_size * pos_ratio)
    B_neg = batch_size - B_pos

    probes, refs, labels = [], [], []

    # Positive pairs (같은 ID)
    for _ in range(B_pos):
        id_ = random.choice(ids)
        paths = id2paths[id_]
        q, *ref = sample_k(paths, k_refs+1)  # 총 k_refs+1장을 뽑아 1장은 질의, 5장은 참조
        probe_img = load_img(q, train_tf if train_mode else eval_tf)
        ref_imgs = torch.stack([load_img(p, train_tf if train_mode else eval_tf) for p in ref], dim=0)
        probes.append(probe_img)
        refs.append(ref_imgs)
        labels.append(1.0)

    # Negative pairs (다른 ID)
    for _ in range(B_neg):
        id_q = random.choice(ids)
        id_r = random.choice(ids)
        while id_r == id_q:
            id_r = random.choice(ids)

        q_path = random.choice(id2paths[id_q])
        ref_paths = sample_k(id2paths[id_r], k_refs)

        probe_img = load_img(q_path, train_tf if train_mode else eval_tf)
        ref_imgs = torch.stack([load_img(p, train_tf if train_mode else eval_tf) for p in ref_paths], dim=0)
        probes.append(probe_img)
        refs.append(ref_imgs)
        labels.append(0.0)

    probe = torch.stack(probes, dim=0).to(device)            # (B,3,H,W)
    refs  = torch.stack(refs, dim=0).to(device)              # (B,K,3,H,W)
    labels = torch.tensor(labels, dtype=torch.float32, device=device)
    return probe, refs, labels

In [ ]:
#---------------------
# 3. 모델: CNN(ResNet18) 임베딩 + 참조(5장) 평균 임베딩 + 코사인 유사도
#---------------------

# CNN -> embadding
class Encoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        # 마지막 FC 교체
        in_features = m.fc.in_features
        m.fc = nn.Identity()
        self.backbone = m
        self.proj = nn.Linear(in_features, out_dim)

    def forward(self, x):           # x: (B,3,H,W)
        h = self.backbone(x)        # (B,512)
        z = F.normalize(self.proj(h), dim=-1)  # (B,out_dim), L2 norm
        return z

# 임베딩 사이 유사도 비교
class Verifier(nn.Module):
    def __init__(self, emb_dim=256, temperature=0.07):
        super().__init__()
        self.enc = Encoder(out_dim=emb_dim)
        self.tau = nn.Parameter(torch.tensor(temperature))   # 학습 가능한 온도

    def forward(self, probe, refs):
        """
        probe: (B,3,H,W)
        refs : (B,K,3,H,W)
        return: logits (B,)
        """
        B, K = refs.shape[0], refs.shape[1]
        # (B*K,3,H,W)로 펼쳐서 인코딩 후 (B,K,D)로 복원
        refs_flat = refs.view(B*K, *refs.shape[2:])
        z_probe = self.enc(probe)                    # (B,D)
        z_refs  = self.enc(refs_flat).view(B, K, -1) # (B,K,D)

        # 참조 임베딩 평균 (또는 median/attention로 바꿀 수 있음)
        z_ref_agg = F.normalize(z_refs.mean(dim=1), dim=-1)  # (B,D)

        # 코사인 유사도 -> 온도 스케일링 -> 로짓
        cos = (z_probe * z_ref_agg).sum(dim=-1)              # (B,)
        logits = cos / torch.clamp(self.tau, min=1e-6)
        return logits


# 같은 사람이면 1 아니면 0
def train_loop(model, train_data, epochs=5, steps_per_epoch=500,
               batch_size=32, k_refs=5, lr=1e-4, pos_ratio=0.5):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs*steps_per_epoch)
    loss_fn = nn.BCEWithLogitsLoss()

    for ep in range(1, epochs+1):
        running = []
        # 🔥 tqdm으로 step 진행률 표시
        for step in trange(1, steps_per_epoch+1, desc=f"Epoch {ep}/{epochs}"):
            probe, refs, labels = make_batch(train_data, batch_size, k_refs, pos_ratio, train_mode=True)
            logits = model(probe, refs)
            loss = loss_fn(logits, labels)

            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()

            running.append(loss.item())
            
        avg_loss = np.mean(running)
        acc, auc, _ = evaluate(model, train_data, n_trials=400, k_refs=k_refs)
        print(f"[Epoch {ep}/{epochs}] loss={avg_loss:.4f} | Train ACC={acc:.3f}, AUC={auc:.3f}")

# 평가 루틴 
@torch.no_grad()
def evaluate(model, id2paths, n_trials=1000, k_refs=5):
    model.eval()
    y_true, y_score = [], []

    ids = list(id2paths.keys())
    for i in trange(n_trials, desc="Evaluating"):
        is_pos = random.random() < 0.5
        if is_pos:
            id_ = random.choice(ids)
            paths = id2paths[id_]
            q, *ref = sample_k(paths, k_refs+1)
            label = 1
        else:
            id_q = random.choice(ids)
            id_r = random.choice(ids)
            while id_r == id_q:
                id_r = random.choice(ids)
            q  = random.choice(id2paths[id_q])
            ref = sample_k(id2paths[id_r], k_refs)
            label = 0

        probe = load_img(q, eval_tf).unsqueeze(0).to(device)
        refs  = torch.stack([load_img(p, eval_tf) for p in ref], 0)\
                    .unsqueeze(0).to(device)
        logit = model(probe, refs)

        y_true.append(label)
        y_score.append(torch.sigmoid(logit).item())

    # ROC 기반 threshold
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    j_scores = tpr - fpr
    best_idx = j_scores.argmax()
    best_threshold = thresholds[best_idx]

    y_pred = [1 if s >= best_threshold else 0 for s in y_score]

    acc = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_score)
    except:
        auc = float("nan")

    model.train()
    return acc, auc, best_threshold



In [ ]:
# ----------------
# 실행
# ----------------

model = Verifier(emb_dim=256, temperature=0.07) # 초기화

train_loop(
    model,
    train_data,
    epochs=10,             # 학습 epoch 수 (필요시 늘리기)
    steps_per_epoch=400,  # epoch마다 몇 step 도는지
    batch_size=32,        # 배치 크기
    k_refs=5,             # 참조 이미지 개수
    lr=2e-4,              # learning rate
    pos_ratio=0.5         # 양성/음성 비율
)

# 테스트셋 평가 
acc, auc, best_th = evaluate(
    model,
    test_data,
    n_trials=1500,  # 테스트셋에서 샘플링 횟수 (늘리면 안정적)
    k_refs=5
)

print(f"[TEST RESULT] ACC: {acc:.3f}, AUC: {auc:.3f}, Best Threshold: {best_th:.3f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
import os

# 1) enc 안에서 정확한 서브모듈을 꺼낸다
#    - enc.backbone: ResNet (fc=Identity → 512차원 벡터)
#    - enc.proj: Linear(512→256)
backbone = deepcopy(model.enc.backbone).eval().cpu()
proj     = deepcopy(model.enc.proj).eval().cpu()

# 2) 추론용 Encoder 래퍼 (한 번만 통과 + L2 normalize)
class Encoder(nn.Module):
    def __init__(self, back, proj):
        super().__init__()
        self.back = back
        self.proj = proj
    def forward(self, x):
        z = self.back(x)      # [B, 512]
        z = self.proj(z)      # [B, 256]
        z = F.normalize(z, dim=1)
        return z

enc = Encoder(backbone, proj).eval().cpu()

# 3) 메타 (학습 때 값으로 맞춰줘)
best_th = 0.72  # evaluate()에서 구한 게 있으면 그 값
meta = {
    "emb_dim": 256,
    "input_size": (224, 224),   # 학습 때 입력 크기
    "mean": [0.5, 0.5, 0.5],    # 학습 전처리 mean
    "std":  [0.5, 0.5, 0.5],    # 학습 전처리 std
    "best_threshold": float(best_th),
}

# 4) 산티 체크 (출력 차원 256 확인)
x = torch.randn(1, 3, meta["input_size"][0], meta["input_size"][1])
with torch.no_grad():
    y = enc(x)
print("encoder output shape:", y.shape)  # → torch.Size([1, 256]) 여야 함

# 5) 저장 (state_dict + TorchScript + meta)
os.makedirs("models", exist_ok=True)
torch.save({"encoder_state": enc.state_dict(), "meta": meta}, "models/encoder.pt")
print("✓ Saved models/encoder.pt")

scripted = torch.jit.trace(enc, x)
scripted.save("models/encoder_ts.pt")
torch.save({"meta": meta}, "models/encoder_meta.pt")
print("✓ Saved models/encoder_ts.pt and models/encoder_meta.pt")


In [ ]:
# ================================================
# ROC / PR / Score Histogram : 실행 즉시 표시 + 저장
# ================================================
import os, numpy as np, matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix

# ----- 한글 폰트 자동 설정 (macOS/Windows/Linux) -----
def set_korean_font():
    candidates = ["AppleGothic", "Malgun Gothic", "NanumGothic"]
    avail = {f.name for f in font_manager.fontManager.ttflist}
    chosen = next((c for c in candidates if c in avail), None)
    if chosen: mpl.rcParams["font.family"] = chosen
    mpl.rcParams["axes.unicode_minus"] = False
set_korean_font()

def run_eval_and_plot_clean(y_true, y_score=None, y_dist=None, bins=50,
                            save=True, out_dir="outputs"):
    assert (y_score is not None) or (y_dist is not None), "y_score 또는 y_dist 필요"
    if y_score is None:
        y_score = 1.0 / (1.0 + np.asarray(y_dist, dtype=float))
    else:
        y_score = np.asarray(y_score, dtype=float)
    y_true = np.asarray(y_true, dtype=int)

    # ---- 지표 ----
    auc = roc_auc_score(y_true, y_score)
    ap  = average_precision_score(y_true, y_score)
    fpr, tpr, thr = roc_curve(y_true, y_score)
    best_t = thr[np.argmax(tpr - fpr)]
    pred   = (y_score >= best_t).astype(int)
    cm     = confusion_matrix(y_true, pred, labels=[0,1])

    print(f"[Validation] ROC-AUC={auc:.3f} | PR-AUC={ap:.3f}")
    print(f"Best threshold={best_t:.3f}")
    print("Confusion Matrix (true 0/1 vs pred 0/1):\n", cm)

    if save:
        os.makedirs(out_dir, exist_ok=True)

    # ---- ROC (제목에 AUC, legend X) ----
    plt.figure(figsize=(7,5))
    plt.plot(fpr, tpr)
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(f"ROC (AUC={auc:.3f})")
    plt.grid(True, alpha=.3)
    if save: plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # ---- PR (제목에 AP, legend X) ----
    p, r, _ = precision_recall_curve(y_true, y_score)
    plt.figure(figsize=(7,5))
    plt.plot(r, p)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"PR (AP={ap:.3f})")
    plt.grid(True, alpha=.3)
    if save: plt.savefig(os.path.join(out_dir, "pr_curve.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # ---- Score Histogram (클래스 범례 유지) ----
    plt.figure(figsize=(7,5))
    plt.hist(y_score[y_true==1], bins=bins, alpha=.6, label="Positive(자필-자필)")
    plt.hist(y_score[y_true==0], bins=bins, alpha=.6, label="Negative(타인/모사)")
    plt.legend(loc="upper right")
    plt.title("유사도(score=1/(1+dist)) 분포")
    plt.xlabel("score"); plt.ylabel("count")
    plt.grid(True, alpha=.3)
    if save: plt.savefig(os.path.join(out_dir, "score_hist.png"), dpi=150, bbox_inches="tight")
    plt.show()

    return {"auc": float(auc), "ap": float(ap), "best_threshold": float(best_t), "confusion_matrix": cm}

# ---------------- 실행(자동) ----------------
# 세션에 y_true와 y_score(또는 y_dist)가 있으면 그대로 사용하고,
# 없으면 데모 데이터를 만들어서라도 즉시 그래프를 띄웁니다.
_glob = globals()
if ('y_true' in _glob) and (('y_score' in _glob) or ('y_dist' in _glob)):
    stats = run_eval_and_plot_clean(_glob['y_true'], y_score=_glob.get('y_score'), y_dist=_glob.get('y_dist'))
else:
    rng = np.random.default_rng(1)
    n_pos, n_neg = 800, 1200
    d_pos = rng.normal(0.22, 0.07, size=n_pos).clip(0, 3)  # 같은 사람: 거리 작음
    d_neg = rng.normal(0.68, 0.12, size=n_neg).clip(0, 3)  # 다른 사람: 거리 큼
    y_true_demo = np.array([1]*n_pos + [0]*n_neg)
    y_dist_demo = np.concatenate([d_pos, d_neg])
    stats = run_eval_and_plot_clean(y_true_demo, y_dist=y_dist_demo)

print(stats)


In [ ]:
# =========================================================
# Probe vs Ref 2장 + Agreement 시각화 (Grad-CAM++) - 미니멀 버전
#   - 텍스트/제목/주석 모두 제거
#   - 히트맵만 출력 (컬러바 옵션)
# =========================================================

import os, random
import numpy as np
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def visualize_pair_campp_minimal_with_title(
    model,
    test_data,
    mode="pos",
    k_pick=12,
    cam_layer="layer3",
    save_path="campp_pair_min.png",
    show_colorbar=False,
    overlay_alpha=0.45
):
    """
    상단 제목 "시각화 - 유사도 히트맵" 추가한 미니멀 버전.
    """
    model.eval().to(device)

    # 1) Probe 선택
    ids = list(test_data.keys())
    probe_id = random.choice(ids)
    while len(test_data[probe_id]) < 1:
        probe_id = random.choice(ids)
    probe_path = random.choice(test_data[probe_id])
    probe_img  = load_img(probe_path, eval_tf)

    # 2) 후보 풀
    if mode == "pos":
        cand_pool = [p for p in test_data[probe_id] if p != probe_path]
        if len(cand_pool) == 0:
            cand_pool = test_data[probe_id]
    else:
        other_ids = [i for i in ids if i != probe_id and len(test_data[i]) > 0]
        cand_pool = test_data[random.choice(other_ids)]
    cand_paths = random.sample(cand_pool, min(k_pick, len(cand_pool))) if len(cand_pool) >= k_pick else random.choices(cand_pool, k=k_pick)

    # 3) Top-1 Ref
    ref_path, _, _ = pick_top1_ref(model, probe_img, cand_paths, eval_tf)
    ref_img = load_img(ref_path, eval_tf)

    # 4) CAM 계산
    with torch.no_grad():
        z_probe = model.enc(probe_img.unsqueeze(0).to(device))
        z_ref   = model.enc(ref_img.unsqueeze(0).to(device))
    cam_probe, _, _ = grad_cam_pp_for_image(model, probe_img.unsqueeze(0), z_ref,   cam_layer=cam_layer)
    cam_ref,   _, _ = grad_cam_pp_for_image(model, ref_img.unsqueeze(0),   z_probe, cam_layer=cam_layer)
    agree = np.minimum(cam_probe, cam_ref)

    # 5) 시각화
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.8))

    axes[0].imshow(denorm_np(probe_img))
    im0 = axes[0].imshow(cam_probe, cmap="jet", alpha=overlay_alpha)
    axes[0].axis("off")
    if show_colorbar: plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    axes[1].imshow(denorm_np(ref_img))
    im1 = axes[1].imshow(cam_ref, cmap="jet", alpha=overlay_alpha)
    axes[1].axis("off")
    if show_colorbar: plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(agree, cmap="jet")
    axes[2].axis("off")
    if show_colorbar: plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    # ✅ 제목 추가
    fig.suptitle("시각화 - 유사도 히트맵", fontsize=14, y=0.95)

    plt.tight_layout(pad=0.05)
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"✅ 저장: {os.path.abspath(save_path)}")
    plt.show()

    return {"probe_path": probe_path, "ref_path": ref_path}


# ---------------- 실행 예시 ----------------
# 👉 같은 사람(양성)으로 시각화
visualize_pair_campp_minimal(
    model,
    test_data,
    mode="pos",
    k_pick=12,
    cam_layer="layer3",
    save_path="campp_pair_min.png",
    show_colorbar=False  # True로 하면 컬러바 숫자 표시
)

# 👉 다른 사람(음성) 예시
# visualize_pair_campp_minimal(model, test_data, mode="neg", k_pick=12, cam_layer="layer3", save_path="campp_pair_min_neg.png")


In [ ]:
# 학습 끝난 뒤 enc = Encoder(backbone=..., proj=...) 가 있다고 가정
meta = {
    "emb_dim": 256,
    "input_size": (224, 224),
    "mean": [0.5, 0.5, 0.5],
    "std": [0.5, 0.5, 0.5],
    "best_threshold": float(best_th),
    "backbone_name": "resnet18",
}

# Eager 저장 (CAM용)
torch.save(
    {"state_dict": enc.state_dict(), "meta": meta},
    "models/encoder_eager.pt"
)
print("✓ saved models/encoder_eager.pt")

# (지금처럼 쓰던 TorchScript/메타 파일은 그대로 유지)
# scripted.save("models/encoder_ts.pt")
# torch.save({"meta": meta}, "models/encoder_meta.pt")


In [ ]:
# =========================================================
# 단일 이미지 Grad-CAM++ 시각화 ('사하구')
# =========================================================
import os
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def visualize_single_campp(
    model,
    img_path,           # 분석할 이미지 경로
    cam_layer="layer3",
    save_path="campp_single.png",
    show_colorbar=True,
    overlay_alpha=0.45
):
    """
    단일 이미지 Grad-CAM++ 히트맵 시각화
    """
    model.eval().to(device)

    # 이미지 로드
    img = load_img(img_path, eval_tf)

    # 임베딩
    with torch.no_grad():
        z = model.enc(img.unsqueeze(0).to(device))

    # Grad-CAM++ (자기 자신 임베딩을 target으로 사용)
    cam, _, _ = grad_cam_pp_for_image(model, img.unsqueeze(0), z, cam_layer=cam_layer)

    # 시각화
    fig, ax = plt.subplots(1, 1, figsize=(5,5))
    ax.imshow(denorm_np(img))
    im = ax.imshow(cam, cmap="jet", alpha=overlay_alpha)
    ax.axis("off")
    if show_colorbar:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle("시각화 - 유사도 히트맵 ('사하구')", fontsize=14)
    plt.tight_layout(pad=0.05)

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"✅ 저장: {os.path.abspath(save_path)}")

    plt.show()


# ---------------- 실행 ----------------
img_path = "./all_crops/IMG_OCR_53_4PR_10551/IMG_OCR_53_4PR_10551_08_사하구.png"

visualize_single_campp(
    model,
    img_path,
    cam_layer="layer3",
    save_path="campp_saha.png",
    show_colorbar=True  # False로 하면 컬러바 제거
)
